# Galileo Brain — Fine-Tuning Qwen3-4B con Unsloth

**9 celle**. Copia-incolla una per una. Dopo la cella 1, riavvia il runtime.

3 fix vs notebook vecchio:
1. Install pinnato: `transformers==4.56.2` + `trl==0.22.2`
2. Modello: `unsloth/Qwen3-4B-Instruct-2507` (text-only pre-quantizzato)
3. `train_on_responses_only` — il modello impara SOLO le risposte, non il system prompt

In [ ]:
# ============================================================
# CELLA 1 — Install (ESATTA da notebook ufficiale Unsloth)
# ============================================================
# DOPO QUESTA CELLA: Runtime > Restart session
# ============================================================

%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10': '0.0.34',
        '2.9': '0.0.33.post1',
        '2.8': '0.0.32.post2'
    }.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

print("--- Install completato. Riavvia il runtime (Runtime > Restart session) ---")

In [ ]:
# ============================================================
# CELLA 2 — Carica modello + LoRA
# ============================================================
# CHECKPOINT: deve stampare Trainable > 0
# ============================================================

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")
assert trainable > 0, "ERRORE: 0 parametri trainabili!"
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# CELLA 3 — Upload dataset
# ============================================================

from google.colab import files
import os

DATASET_PATH = "galileo-brain-poc.jsonl"
if not os.path.exists(DATASET_PATH):
    print("Carica galileo-brain-poc.jsonl...")
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

with open(DATASET_PATH) as f:
    num_lines = sum(1 for _ in f)
print(f"Dataset: {DATASET_PATH} — {num_lines} esempi")

In [ ]:
# ============================================================
# CELLA 4 — Prepara dataset con chat template
# ============================================================

from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

dataset = load_dataset("json", data_files={"train": DATASET_PATH}, split="train")
print(f"Colonne: {dataset.column_names}, Righe: {len(dataset)}")

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"\nEsempio formattato (500 chars):\n{dataset[0]['text'][:500]}")

In [ ]:
# ============================================================
# CELLA 5 — SFTTrainer + train_on_responses_only
# ============================================================
# FIX CRITICO: senza train_on_responses_only il modello
# spreca token imparando a ripetere il system prompt
# ============================================================

from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
        output_dir="galileo-brain-output",
        save_steps=50,
        save_total_limit=2,
    ),
)

# Masking: loss calcolata SOLO sulle risposte assistant
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Trainer pronto. Loss solo su risposte assistant.")

In [ ]:
# ============================================================
# CELLA 6 — Training
# ============================================================
# CHECKPOINT: loss finale deve scendere sotto ~0.5
# ============================================================

import time

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\n{'='*50}")
print(f"TRAINING COMPLETATO!")
print(f"Tempo: {elapsed/60:.1f} min")
print(f"Loss finale: {stats.training_loss:.4f}")
print(f"GPU peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# CELLA 7 — Test inferenza
# ============================================================
# CHECKPOINT: i 3 test devono produrre JSON valido
# ============================================================

import json
FastLanguageModel.for_inference(model)

# System prompt dal dataset
with open(DATASET_PATH) as f:
    first = json.loads(f.readline())
SYSTEM = first["messages"][0]["content"]

def test(msg):
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": msg}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs, max_new_tokens=512,
        temperature=0.1, top_p=0.95,
    )
    resp = tokenizer.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    try:
        return json.loads(resp)
    except:
        return {"raw": resp}

tests = [
    "[CONTESTO]\ntab: simulator\ncomponenti: [led1]\nfili: 1\n\n[MESSAGGIO]\nAvvia la simulazione",
    "[CONTESTO]\ntab: simulator\ncomponenti: []\nfili: 0\n\n[MESSAGGIO]\nMetti un LED rosso",
    "[CONTESTO]\ntab: simulator\ncomponenti: [resistor1]\n\n[MESSAGGIO]\nCos'e' un resistore?",
]

for t in tests:
    r = test(t)
    intent = r.get("intent", "?") if isinstance(r, dict) else "?"
    ok = "raw" not in r
    print(f"{'\u2705' if ok else '\u26a0\ufe0f'} intent={intent} | {str(r)[:120]}")

In [ ]:
# ============================================================
# CELLA 8 — Salva LoRA + GGUF
# ============================================================

# LoRA adapter
model.save_pretrained("galileo-brain-lora")
tokenizer.save_pretrained("galileo-brain-lora")
print("LoRA salvato.")

# GGUF per Ollama
model.save_pretrained_gguf(
    "galileo-brain-gguf", tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF q4_k_m salvato.")

In [ ]:
# ============================================================
# CELLA 9 — Download GGUF
# ============================================================

from google.colab import files
import os

gguf_dir = "galileo-brain-gguf"
for f in os.listdir(gguf_dir):
    if f.endswith(".gguf"):
        size = os.path.getsize(os.path.join(gguf_dir, f)) / 1e9
        print(f"Scaricando {f} ({size:.2f} GB)...")
        files.download(os.path.join(gguf_dir, f))